# Mahindra & Mahindra — Financial Model (Python)

Same structure as the Excel version: revenue buildup → P&L → PP&E/Depreciation →
Working Capital → Debt schedule (circular) → WACC → DCF.

Historical data (FY2022–FY2026) is hardcoded from the annual reports in
`Mahindra_Mahindra_Historical_Financial_Model_Base.xlsx`. Everything from
FY2027–FY2031 is forecast.

In [1]:
import numpy as np
import pandas as pd

file_path = r"C:\Users\Mikey\Downloads\Mahindra_Mahindra_Historical_Financial_Model_Base.xlsx"

hist_years = ["FY2022", "FY2023", "FY2024", "FY2025", "FY2026"]
forecast_years = ["FY2027", "FY2028", "FY2029", "FY2030", "FY2031"]

## 1. Revenue Buildup (by segment, scenario-based)

In [ ]:
segment_revenue = pd.read_excel(file_path, sheet_name="Segment revenue", header=3, nrows=4).set_index("Segment")
segment_revenue

,FY2022,FY2023,FY2024,FY2025,FY2026
Segment,,,,,
Automotive,36705.85,59502.69,75488.31,90424.08,117363.87
Farm Equipment,26828.22,31690.61,33398.87,35315.81,42397.44
Financial Services,11209.23,12554.28,15652.02,18295.83,20817.69
Other Businesses,15427.27,17520.97,14539.07,15175.10,18059.55


In [ ]:
growth = segment_revenue.pct_change(axis=1)
growth_clean = growth.drop(columns="FY2022")

growth_clean["Average_growth_rate"] = growth_clean.mean(axis=1)
growth_clean["CAGR_FY22_FY26"] = (segment_revenue["FY2026"] / segment_revenue["FY2022"]) ** (1 / 4) - 1
growth_clean["CAGR_FY23_FY26"] = (segment_revenue["FY2026"] / segment_revenue["FY2023"]) ** (1 / 3) - 1

growth_clean

,FY2023,FY2024,FY2025,FY2026,Average_growth_rate,CAGR_FY22_FY26,CAGR_FY23_FY26
Segment,,,,,,,
Automotive,0.621068,0.268654,0.197855,0.297927,0.346376,0.337211,0.254101
Farm Equipment,0.181242,0.053904,0.057395,0.200523,0.123266,0.121210,0.101885
Financial Services,0.119995,0.246748,0.168912,0.137838,0.168373,0.167385,0.183623
Other Businesses,0.135714,-0.170190,0.043746,0.190078,0.049837,0.040170,0.010143


In [4]:
# Scenario assumptions - your judgment call based on the growth table above +
# industry research. Bull/Bear are +/-3pp around Base. Edit these directly.
base = {
    "Automotive": 0.1151,
    "Farm Equipment": 0.0755,
    "Financial Services": 0.1348,
    "Other Businesses": 0.05,
}
scenarios = {
    "Base": base,
    "Bull": {seg: g + 0.03 for seg, g in base.items()},
    "Bear": {seg: g - 0.03 for seg, g in base.items()},
}
scenarios

{'Base': {'Automotive': 0.1151,
  'Farm Equipment': 0.0755,
  'Financial Services': 0.1348,
  'Other Businesses': 0.05},
 'Bull': {'Automotive': 0.1451,
  'Farm Equipment': 0.1055,
  'Financial Services': 0.1648,
  'Other Businesses': 0.08},
 'Bear': {'Automotive': 0.0851,
  'Farm Equipment': 0.0455,
  'Financial Services': 0.1048,
  'Other Businesses': 0.020000000000000004}}

In [5]:
def build_revenue_forecast(segment_revenue, scenario_name, scenarios, forecast_years):
    """Project each segment forward at a flat CAGR, then sum to Total Revenue."""
    assumptions = scenarios[scenario_name]
    rev = segment_revenue.copy()
    # index.to_series().map(...) aligns the growth rate to each segment by its label,
    # instead of by row position — this is what .iloc-based code was missing.
    growth_rate = rev.index.to_series().map(assumptions)

    prev_col = "FY2026"
    for year in forecast_years:
        rev[year] = rev[prev_col] * (1 + growth_rate)
        prev_col = year

    rev.loc["Total Revenue"] = rev.sum()
    return rev


revenue_base = build_revenue_forecast(segment_revenue, "Base", scenarios, forecast_years)
revenue_bull = build_revenue_forecast(segment_revenue, "Bull", scenarios, forecast_years)
revenue_bear = build_revenue_forecast(segment_revenue, "Bear", scenarios, forecast_years)

revenue_base

,FY2022,FY2023,FY2024,FY2025,FY2026,FY2027,FY2028,FY2029,FY2030,FY2031
Segment,,,,,,,,,,
Automotive,36705.85,59502.69,75488.31,90424.08,117363.87,130872.451437,145935.870597,162733.089303,181463.667882,202350.136055
Farm Equipment,26828.22,31690.61,33398.87,35315.81,42397.44,45598.446720,49041.129447,52743.734721,56725.886692,61008.691137
Financial Services,11209.23,12554.28,15652.02,18295.83,20817.69,23623.914612,26808.418302,30422.193089,34523.104717,39176.819233
Other Businesses,15427.27,17520.97,14539.07,15175.10,18059.55,18962.527500,19910.653875,20906.186569,21951.495897,23049.070692
Total Revenue,90170.57,121268.55,139078.27,159210.82,198638.55,219057.340269,241696.072221,266805.203681,294664.155188,325584.717118


## 2. P&L history + cost ratio assumptions

In [6]:
pnl_hist = pd.read_excel(file_path, sheet_name="P&L", header=2).set_index("₹ crore")
pnl_hist

,FY2022,FY2023,FY2024,FY2025,FY2026
₹ crore,,,,,
Revenue from operations,90170.57,121268.55,138279.30,158749.75,197792.78
Income from investments related to subsidiaries/associates/JVs,0.00,0.00,798.97,461.07,845.77
Income from operations,90170.57,121268.55,139078.27,159210.82,198638.55
Other income,934.51,1206.49,2176.42,2181.05,3445.79
Total income,91105.08,122475.04,141254.69,161391.87,202084.34
Cost of materials consumed,46265.48,68477.97,77848.82,88111.05,113553.14
Purchases of stock-in-trade,6399.37,7541.90,7221.25,7643.85,8670.45
Changes in inventories of FG/ST/WIP,-861.66,-2032.31,-1455.32,-12.18,698.79
Employee benefits expense,8386.74,9677.95,10624.33,11126.17,12730.62


In [ ]:
def ratio(row):
    return pnl_hist.loc[row, hist_years] / pnl_hist.loc["Income from operations", hist_years]

material_ratio = ratio("Cost of materials consumed")
purchase_ratio = ratio("Purchases of stock-in-trade")
inv_change_ratio = ratio("Changes in inventories of FG/ST/WIP")
employee_ratio = ratio("Employee benefits expense")
other_exp_ratio = ratio("Other expenses")

pd.DataFrame({
    "Material cost %": material_ratio,
    "Purchases %": purchase_ratio,
    "Inventory change %": inv_change_ratio,
    "Employee cost %": employee_ratio,
    "Other expenses %": other_exp_ratio,})

,Material cost %,Purchases %,Inventory change %,Employee cost %,Other expenses %
FY2022,0.513088,0.070970,-0.009556,0.093010,0.171375
FY2023,0.564680,0.062192,-0.016759,0.079806,0.142805
FY2024,0.559748,0.051922,-0.010464,0.076391,0.143188
FY2025,0.553424,0.048011,-0.000077,0.069883,0.136881
FY2026,0.571657,0.043649,0.003518,0.064089,0.127992


In [24]:
# Forward assumption = FY2026 (latest actual) ratio. Swap to .mean() over
# hist_years if you'd rather smooth across the 5-year history instead.
assum_material = material_ratio["FY2026"]
assum_purchase = purchase_ratio["FY2026"]
assum_inv_change = inv_change_ratio["FY2026"]
assum_employee = employee_ratio["FY2026"]
assum_other_exp = other_exp_ratio["FY2026"]

# Effective tax rate: (current tax + deferred tax) / PBT, averaged over history
eff_tax_rate = (
    -(pnl_hist.loc["Current tax", hist_years] + pnl_hist.loc["Deferred tax", hist_years])
    / pnl_hist.loc["Profit before tax", hist_years]
).mean()

print("Effective tax rate (avg FY22-26):", round(eff_tax_rate, 4))

Effective tax rate (avg FY22-26): 0.2343


## 3. PP&E / Depreciation and Working Capital schedules

Both are needed *before* the P&L forecast function is called, because Depreciation
and Finance costs are inputs to the P&L, not outputs of it.

In [27]:
bs_hist = pd.read_excel(file_path, sheet_name="Balance_Sheet", header=2).set_index("₹ crore")

# Net fixed assets = PPE + CWIP (the two capitalised, depreciating asset lines)
ppe_lines = ["Property, plant and equipment", "Capital work-in-progress"]
net_ppe_hist = bs_hist.loc[ppe_lines, hist_years].sum()

# Capex, pulled from the Cash_Flow sheet's "Capex - PPE and other intangibles" line
# (sign-flipped to positive; the sheet stores it as a cash outflow)
cash_flow_hist = pd.read_excel(file_path, sheet_name="Cash_Flow", header=2).set_index("₹ crore")

# Capex is stored as a negative (cash outflow) — flip sign to positive for our schedule
capex_hist = -cash_flow_hist.loc["Capex - PPE and other intangibles", hist_years]
capex_pct_sales = capex_hist / pnl_hist.loc["Income from operations", hist_years]

assum_capex_pct = capex_pct_sales["FY2026"]
# Depreciation as a single blended rate on opening net PP&E (proxy for a full
# asset-by-asset SLM schedule — good enough at this level of granularity since
# we only have net PP&E, not a gross asset breakup, from the base file)
assum_dep_pct_of_ppe = pnl_hist.loc["Depreciation, amortisation and impairment", "FY2026"] / net_ppe_hist["FY2026"]

print("Capex % of sales (FY2026):", round(assum_capex_pct, 4))
print("Depreciation % of opening PP&E (FY2026):", round(assum_dep_pct_of_ppe, 4))

Capex % of sales (FY2026): 0.0484
Depreciation % of opening PP&E (FY2026): 0.2333


In [ ]:
def build_ppe_schedule(revenue_row, opening_ppe_fy26, forecast_years):
    capex = assum_capex_pct * revenue_row
    ppe = pd.Series(index=forecast_years, dtype=float)
    dep = pd.Series(index=forecast_years, dtype=float)

    opening = opening_ppe_fy26
    for year in forecast_years:
        dep[year] = assum_dep_pct_of_ppe * opening
        ppe[year] = opening + capex[year] - dep[year]
        opening = ppe[year]

    return ppe, dep, capex


ppe_fcst, dep_fcst, capex_fcst = build_ppe_schedule(revenue_base.loc["Total Revenue", forecast_years], net_ppe_hist["FY2026"], forecast_years)

pd.DataFrame({"Net PP&E": ppe_fcst, "Depreciation": dep_fcst, "Capex": capex_fcst})

,Net PP&E,Depreciation,Capex
FY2027,34651.159685,7322.020000,10592.829685
FY2028,38253.514999,8085.202499,11687.557813
FY2029,42229.515632,8925.744993,12901.745626
FY2030,46624.949877,9853.470661,14248.904905
FY2031,51489.999246,10879.063347,15744.112716


In [ ]:
# Working capital: days-based, same approach as your Excel WC sheet
# (Inventory days, Receivable days, Payable days off COGS/Sales, held flat at FY2026 level)
cogs_hist = (
    pnl_hist.loc["Cost of materials consumed", hist_years]
    + pnl_hist.loc["Purchases of stock-in-trade", hist_years]
    + pnl_hist.loc["Changes in inventories of FG/ST/WIP", hist_years])

inventory_days = bs_hist.loc["Inventories", hist_years] / (cogs_hist / 365)
receivable_days = bs_hist.loc["Current trade receivables", hist_years] / (
    pnl_hist.loc["Income from operations", hist_years] / 365)
payable_days = (
    bs_hist.loc["Trade payables - MSME", hist_years] + bs_hist.loc["Trade payables - other creditors", hist_years]
) / (cogs_hist / 365)

assum_inv_days = inventory_days["FY2026"]
assum_recv_days = receivable_days["FY2026"]
assum_pay_days = payable_days["FY2026"]

pd.DataFrame({
    "Inventory days": inventory_days,
    "Receivable days": receivable_days,
    "Payable days": payable_days,
})

,Inventory days,Receivable days,Payable days
FY2022,81.702967,25.801010,134.129592
FY2023,83.149979,21.153278,117.587550
FY2024,81.152208,19.576610,112.037154
FY2025,77.507611,18.981690,117.629669
FY2026,65.456755,16.655672,112.504855


In [12]:
def build_wc_schedule(revenue_row, cogs_row, forecast_years):
    inv = assum_inv_days / 365 * cogs_row
    recv = assum_recv_days / 365 * revenue_row
    pay = assum_pay_days / 365 * cogs_row
    nwc = inv + recv - pay
    return inv, recv, pay, nwc

## 4. P&L forecast function

In [13]:
def build_pnl_forecast(revenue_row, pnl_hist, forecast_years, depreciation_row, interest_row):
    """
    revenue_row: Total Revenue for FY2027-31 from the revenue buildup
    depreciation_row / interest_row: FY2027-31 series from their own schedules
    (interest_row will come from the debt schedule, solved iteratively — see step 5)
    """
    pnl = pd.DataFrame(index=pnl_hist.index, columns=forecast_years, dtype=float)

    pnl.loc["Income from operations"] = revenue_row
    pnl.loc["Revenue from operations"] = revenue_row  # simplification: no separate JV income forecast line
    pnl.loc["Other income"] = pnl_hist.loc["Other income", "FY2026"]  # held flat; refine later if you want
    pnl.loc["Total income"] = pnl.loc["Income from operations"] + pnl.loc["Other income"]

    pnl.loc["Cost of materials consumed"] = assum_material * pnl.loc["Income from operations"]
    pnl.loc["Purchases of stock-in-trade"] = assum_purchase * pnl.loc["Income from operations"]
    pnl.loc["Changes in inventories of FG/ST/WIP"] = assum_inv_change * pnl.loc["Income from operations"]
    pnl.loc["Employee benefits expense"] = assum_employee * pnl.loc["Income from operations"]
    pnl.loc["Other expenses"] = assum_other_exp * pnl.loc["Income from operations"]

    pnl.loc["Depreciation, amortisation and impairment"] = depreciation_row
    pnl.loc["Finance costs"] = interest_row
    pnl.loc["Loss from investments related to subsidiaries/associates/JVs"] = 0.0

    pnl.loc["Total expenses"] = pnl.loc[[
        "Cost of materials consumed", "Purchases of stock-in-trade",
        "Changes in inventories of FG/ST/WIP", "Employee benefits expense",
        "Finance costs", "Depreciation, amortisation and impairment",
        "Loss from investments related to subsidiaries/associates/JVs", "Other expenses",
    ]].sum()

    pnl.loc["PBT before exceptional items and associates/JVs"] = pnl.loc["Total income"] - pnl.loc["Total expenses"]
    pnl.loc["Exceptional items (net)"] = 0.0

    # Share of profit of associates/JVs: grown in line with revenue growth (simple proxy)
    jv_growth = revenue_row / revenue_row.shift(1).fillna(pnl_hist.loc["Income from operations", "FY2026"])
    pnl.loc["Share of profit of associates and JVs (net)"] = (
        pnl_hist.loc["Share of profit of associates and JVs (net)", "FY2026"] * jv_growth.cumprod()
    )

    pnl.loc["Profit before tax"] = (
        pnl.loc["PBT before exceptional items and associates/JVs"]
        + pnl.loc["Exceptional items (net)"]
        + pnl.loc["Share of profit of associates and JVs (net)"]
    )

    total_tax = pnl.loc["Profit before tax"] * eff_tax_rate
    pnl.loc["Current tax"] = -total_tax
    pnl.loc["Deferred tax"] = 0.0
    pnl.loc["Profit for the year"] = pnl.loc["Profit before tax"] - total_tax

    # Split PAT between owners and NCI using the FY2026 historical proportion
    nci_share = pnl_hist.loc["PAT attributable to NCI", "FY2026"] / pnl_hist.loc["Profit for the year", "FY2026"]
    pnl.loc["PAT attributable to NCI"] = pnl.loc["Profit for the year"] * nci_share
    pnl.loc["PAT attributable to owners"] = pnl.loc["Profit for the year"] * (1 - nci_share)

    return pnl

## 5. Debt schedule with circularity (Balance Sheet + Cash Flow)

**Why this is circular:** Finance costs depend on the average debt balance for the
year. The debt balance is a *plug* — it moves to keep cash at/above a minimum
operating level, after funding capex, working-capital changes and dividends. But
cash available for debt repayment depends on Profit for the year, which itself
depends on Finance costs. So: Interest → PAT → Cash → Debt → Interest, round and
round.

In Excel this is exactly the circular-reference warning you get with a revolver
schedule (and why models often need iterative calculation switched on). In Python
we solve it explicitly with **fixed-point iteration**: guess an interest number,
build everything downstream, recompute interest from the new debt balance, repeat
until it stops changing. It converges in a handful of passes because the system
is close to linear.

In [14]:
min_cash = bs_hist.loc["Cash and cash equivalents", "FY2026"]  # floor: don't let cash fall below FY2026 actual

opening_debt_fy26 = bs_hist.loc["Non-current borrowings", "FY2026"] + bs_hist.loc["Current borrowings", "FY2026"]
opening_cash_fy26 = bs_hist.loc["Cash and cash equivalents", "FY2026"]
opening_equity_fy26 = bs_hist.loc["Total equity", "FY2026"]

# Back-solve an implied blended interest rate from FY2026 actuals:
# Finance costs / average debt balance over FY2025-FY2026
avg_debt_fy26 = (
    (bs_hist.loc["Non-current borrowings", "FY2025"] + bs_hist.loc["Current borrowings", "FY2025"])
    + opening_debt_fy26
) / 2
interest_rate_on_debt = pnl_hist.loc["Finance costs", "FY2026"] / avg_debt_fy26

# Dividend payout ratio: (dividends paid to owners + NCI) / Profit for the year, FY2026
# Figures below are from the Cash_Flow sheet's financing-activities section.
dividend_paid_fy26 = 2820.65 + 555.66
payout_ratio = dividend_paid_fy26 / pnl_hist.loc["Profit for the year", "FY2026"]

print("Implied interest rate on debt:", round(interest_rate_on_debt, 4))
print("Dividend payout ratio:", round(payout_ratio, 4))

Implied interest rate on debt: 0.0765
Dividend payout ratio: 0.1813


In [15]:
def solve_forecast(scenario_revenue, forecast_years, iterations=12, tol=1e-3):
    revenue_row = scenario_revenue.loc["Total Revenue", forecast_years]

    # Initial guess: flat interest at the FY2026 level
    interest_guess = pd.Series(pnl_hist.loc["Finance costs", "FY2026"], index=forecast_years)

    for it in range(iterations):
        pnl_fcst = build_pnl_forecast(revenue_row, pnl_hist, forecast_years, dep_fcst, interest_guess)

        cogs_fcst = (
            pnl_fcst.loc["Cost of materials consumed"]
            + pnl_fcst.loc["Purchases of stock-in-trade"]
            + pnl_fcst.loc["Changes in inventories of FG/ST/WIP"]
        )
        inv_fcst, recv_fcst, pay_fcst, nwc_fcst = build_wc_schedule(revenue_row, cogs_fcst, forecast_years)

        nwc_hist_fy26 = (
            bs_hist.loc["Inventories", "FY2026"]
            + bs_hist.loc["Current trade receivables", "FY2026"]
            - bs_hist.loc["Trade payables - MSME", "FY2026"]
            - bs_hist.loc["Trade payables - other creditors", "FY2026"]
        )
        nwc_series = pd.concat([pd.Series({"FY2026": nwc_hist_fy26}), nwc_fcst])
        change_in_nwc = nwc_series.diff().dropna()  # positive = cash outflow (NWC grew)

        # Equity roll-forward: opening equity + Profit for the year - dividends
        dividends = payout_ratio * pnl_fcst.loc["Profit for the year"]
        equity = pd.Series(index=forecast_years, dtype=float)
        opening_eq = opening_equity_fy26
        for year in forecast_years:
            equity[year] = opening_eq + pnl_fcst.loc["Profit for the year", year] - dividends[year]
            opening_eq = equity[year]

        # Cash generated before financing = PAT + D&A (non-cash) - capex - change in NWC - dividends
        cash_from_ops = pnl_fcst.loc["Profit for the year"] + dep_fcst - change_in_nwc
        cash_before_financing = cash_from_ops - capex_fcst - dividends

        # Debt plug: draw only what's needed to keep cash at the floor
        debt = pd.Series(index=forecast_years, dtype=float)
        cash = pd.Series(index=forecast_years, dtype=float)
        opening_debt = opening_debt_fy26
        opening_csh = opening_cash_fy26
        for year in forecast_years:
            cash_pre_debt = opening_csh + cash_before_financing[year]
            required_debt_draw = max(0.0, min_cash - cash_pre_debt)
            debt[year] = opening_debt + required_debt_draw
            cash[year] = cash_pre_debt + required_debt_draw
            opening_debt = debt[year]
            opening_csh = cash[year]

        # Recompute interest from the new average debt balance and check convergence
        avg_debt = (pd.concat([pd.Series({"FY2026": opening_debt_fy26}), debt]).shift(1).dropna() + debt) / 2
        new_interest_guess = interest_rate_on_debt * avg_debt

        delta = (new_interest_guess - interest_guess).abs().max()
        interest_guess = new_interest_guess
        if delta < tol:
            break

    return {
        "pnl": pnl_fcst, "inventory": inv_fcst, "receivables": recv_fcst, "payables": pay_fcst,
        "nwc": nwc_fcst, "change_in_nwc": change_in_nwc, "equity": equity, "dividends": dividends,
        "debt": debt, "cash": cash, "ppe": ppe_fcst, "depreciation": dep_fcst, "capex": capex_fcst,
        "interest": interest_guess, "iterations_run": it + 1,
    }

In [16]:
result_base = solve_forecast(revenue_base, forecast_years)
result_bull = solve_forecast(revenue_bull, forecast_years)
result_bear = solve_forecast(revenue_bear, forecast_years)

print(f"Base case converged in {result_base['iterations_run']} iterations")
result_base["pnl"].loc[["Income from operations", "Profit before tax", "Profit for the year"]]

Base case converged in 2 iterations


,FY2027,FY2028,FY2029,FY2030,FY2031
₹ crore,,,,,
Income from operations,219057.340269,241696.072221,266805.203681,294664.155188,325584.717118
Profit before tax,29799.466722,33541.078300,37696.905166,42312.724959,47439.895799
Profit for the year,22818.441494,25683.517762,28865.772434,32400.258969,36326.303987


In [17]:
pd.DataFrame({
    "Debt": result_base["debt"],
    "Cash": result_base["cash"],
    "Net debt": result_base["debt"] - result_base["cash"],
    "Interest expense": result_base["interest"],
    "Total Equity": result_base["equity"],
})

,Debt,Cash,Net debt,Interest expense,Total Equity
FY2027,129614.5,20399.621726,109214.878274,9913.67971,128172.220358
FY2028,129614.5,38596.840982,91017.659018,9913.67971,149199.048812
FY2029,129614.5,59110.017810,70504.482190,9913.67971,172831.156017
FY2030,129614.5,82191.270164,47423.229836,9913.67971,199356.910531
FY2031,129614.5,108121.623972,21492.876028,9913.67971,229096.877205


## 6. WACC (CAPM cost of equity + cost of debt from the schedule)

In [18]:
# Editable macro assumptions — swap these for current market data whenever you refresh the model
risk_free_rate = 0.0685   # ~10yr India G-Sec yield
market_return = 0.12      # long-run Sensex/Nifty CAGR assumption
beta = 0.95                # M&M's 5yr beta vs Sensex (public data point — refine with your own regression if you want)
tax_rate = eff_tax_rate

cost_of_equity = risk_free_rate + beta * (market_return - risk_free_rate)
cost_of_debt_pretax = interest_rate_on_debt          # solved above from FY2026 actuals
cost_of_debt_after_tax = cost_of_debt_pretax * (1 - tax_rate)

E = opening_equity_fy26   # book value of equity, FY2026
D = opening_debt_fy26     # book value of debt, FY2026
V = E + D

wacc = (E / V) * cost_of_equity + (D / V) * cost_of_debt_after_tax

print("Cost of Equity (CAPM):", round(cost_of_equity, 4))
print("Cost of Debt (pre-tax):", round(cost_of_debt_pretax, 4))
print("Cost of Debt (after-tax):", round(cost_of_debt_after_tax, 4))
print("Weight of Equity:", round(E/V, 4), " Weight of Debt:", round(D/V, 4))
print("WACC:", round(wacc, 4))

Cost of Equity (CAPM): 0.1174
Cost of Debt (pre-tax): 0.0765
Cost of Debt (after-tax): 0.0586
Weight of Equity: 0.4579  Weight of Debt: 0.5421
WACC: 0.0855


## 7. DCF Valuation (FCFF method)

In [19]:
def run_dcf(result, wacc, tax_rate, terminal_growth, forecast_years, opening_cash, opening_debt):
    pnl_fcst = result["pnl"]
    ebit = pnl_fcst.loc["Profit before tax"] + pnl_fcst.loc["Finance costs"] - pnl_fcst.loc["Other income"]
    nopat = ebit * (1 - tax_rate)

    fcff = nopat + result["depreciation"] - result["capex"] - result["change_in_nwc"]

    n_years = len(forecast_years)
    discount_factors = pd.Series([(1 + wacc) ** -(i + 1) for i in range(n_years)], index=forecast_years)
    pv_fcff = fcff * discount_factors

    terminal_value = fcff[forecast_years[-1]] * (1 + terminal_growth) / (wacc - terminal_growth)
    pv_terminal_value = terminal_value * discount_factors[forecast_years[-1]]

    enterprise_value = pv_fcff.sum() + pv_terminal_value
    equity_value = enterprise_value - opening_debt + opening_cash

    return {
        "ebit": ebit, "nopat": nopat, "fcff": fcff, "discount_factors": discount_factors,
        "pv_fcff": pv_fcff, "terminal_value": terminal_value, "pv_terminal_value": pv_terminal_value,
        "enterprise_value": enterprise_value, "equity_value": equity_value,
    }


terminal_growth = 0.05   # keep <= long-run nominal GDP growth as a sanity check
shares_outstanding = 111.76  # crore shares — update to the latest count if needed

dcf_base = run_dcf(result_base, wacc, tax_rate, terminal_growth, forecast_years, opening_cash_fy26, opening_debt_fy26)
value_per_share_base = dcf_base["equity_value"] / shares_outstanding

print("Enterprise Value (₹ cr):", round(dcf_base["enterprise_value"], 1))
print("Equity Value (₹ cr):", round(dcf_base["equity_value"], 1))
print("Value per share (₹):", round(value_per_share_base, 1))

Enterprise Value (₹ cr): 854957.4
Equity Value (₹ cr): 729635.2
Value per share (₹): 6528.6


In [20]:
pd.DataFrame({
    "EBIT": dcf_base["ebit"],
    "NOPAT": dcf_base["nopat"],
    "FCFF": dcf_base["fcff"],
    "Discount factor": dcf_base["discount_factors"],
    "PV of FCFF": dcf_base["pv_fcff"],
})

,EBIT,NOPAT,FCFF,Discount factor,PV of FCFF
FY2027,36267.356432,27771.119484,25197.280852,0.921218,23212.183451
FY2028,40008.968010,30636.195752,27806.586553,0.848642,23597.843391
FY2029,44164.794876,33818.450424,30699.520047,0.781784,24000.403231
FY2030,48780.614669,37352.936959,33908.434799,0.720194,24420.638423
FY2031,53907.785510,41278.981977,37469.369112,0.663455,24859.246933


In [21]:
# Run all three scenarios through the same DCF for a quick comparison
dcf_results = {}
for name, result in [("Base", result_base), ("Bull", result_bull), ("Bear", result_bear)]:
    d = run_dcf(result, wacc, tax_rate, terminal_growth, forecast_years, opening_cash_fy26, opening_debt_fy26)
    dcf_results[name] = {
        "Enterprise Value": d["enterprise_value"],
        "Equity Value": d["equity_value"],
        "Value per share": d["equity_value"] / shares_outstanding,
    }

pd.DataFrame(dcf_results).T

,Enterprise Value,Equity Value,Value per share
Base,8.549574e+05,729635.150990,6528.589397
Bull,1.018145e+06,892822.765037,7988.750582
Bear,7.086716e+05,583349.384939,5219.661640


In [22]:
# Sensitivity table: Value per share vs WACC and Terminal Growth (Base case)
wacc_range = np.arange(wacc - 0.02, wacc + 0.025, 0.01)
tg_range = np.arange(terminal_growth - 0.02, terminal_growth + 0.025, 0.01)

sensitivity = pd.DataFrame(
    index=[f"{w:.2%}" for w in wacc_range],
    columns=[f"{g:.2%}" for g in tg_range],
)
for w in wacc_range:
    for g in tg_range:
        if w <= g:
            continue  # Gordon growth formula breaks down when WACC <= terminal growth
        d = run_dcf(result_base, w, tax_rate, g, forecast_years, opening_cash_fy26, opening_debt_fy26)
        sensitivity.loc[f"{w:.2%}", f"{g:.2%}"] = round(d["equity_value"] / shares_outstanding, 1)

sensitivity

,3.00%,4.00%,5.00%,6.00%,7.00%
6.55%,7094.6,9964.1,16531.4,46895.0,NaN
7.55%,5255.4,6805.1,9569.3,15895.8,45145.7
8.55%,4079.8,5035.2,6528.6,9192.4,15288.9
9.55%,3263.9,3903.7,4824.7,6264.3,8832.2
10.55%,2664.7,3118.3,3735.3,4623.5,6011.7


## Notes / where to take this next

- **Depreciation** uses a single blended rate on opening PP&E rather than an
  asset-by-asset SLM schedule (your old Excel model had gross PP&E broken out by
  asset class with useful lives — the new base file only has net PP&E, so this
  is a simplification). If you want asset-level granularity back, add a Gross
  PP&E breakup sheet and rebuild `build_ppe_schedule` around it.
- **Beta** is a placeholder (0.95) — your old workbook derived it from a 1-year
  daily-return regression of M&M vs. the Sensex. That's a good next step if you
  want a data-driven beta instead of a public estimate.
- **Terminal growth (5%) is close to WACC (~8.5%)** in the Base case, which makes
  the Gordon Growth terminal value dominate total valuation — worth stress-testing
  with the sensitivity table above before trusting the headline number.
- Everything is a function of a handful of named variables at the top of each
  section (`assum_material`, `interest_rate_on_debt`, `terminal_growth`, etc.) —
  change those and re-run the notebook top to bottom to see the whole model update.